In [1]:
# --- Portable data location (added by macos-native/tools/mac_python_paths.py) ---------------------
# RAMAN_HANDOFF_DIR is the hand-off folder laser_test/build/Desktop_Qt_6_6_3_MSVC2019_64bit-Debug,
# found by walking up from this file (a notebook: from its working folder). To use another copy,
# set the RAMAN_HANDOFF_DIR environment variable to that folder.
import os as _os


def _raman_handoff_dir():
    if _os.environ.get("RAMAN_HANDOFF_DIR"):
        return _os.path.abspath(_os.environ["RAMAN_HANDOFF_DIR"])
    start = _os.path.dirname(_os.path.abspath(globals()["__file__"])) if "__file__" in globals() else _os.getcwd()
    folder = start
    while True:
        candidate = _os.path.join(folder, "laser_test", "build", "Desktop_Qt_6_6_3_MSVC2019_64bit-Debug")
        if _os.path.isdir(candidate):
            return candidate
        parent = _os.path.dirname(folder)
        if parent == folder:
            raise FileNotFoundError("laser_test/build/Desktop_Qt_6_6_3_MSVC2019_64bit-Debug not found above "
                                    + start + "; set the RAMAN_HANDOFF_DIR environment variable")
        folder = parent


RAMAN_HANDOFF_DIR = _raman_handoff_dir()
# ---------------------------------------------------------------------------------------------------
"""
run_all.py
"""

import os
import re
import sys
import json
import csv
import subprocess
import time
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Optional, Tuple

# =========================================================
# CONFIG: which scripts/notebooks to run
# =========================================================
# Each entry is (display_name, path_to_python_or_notebook_file).
# Both .py and .ipynb are supported.
MODEL_SCRIPTS: List[Tuple[str, str]] = [
    ("PCA-SVM",         f"{RAMAN_HANDOFF_DIR}/Untitled-4_svm.ipynb"),
    ("CNN (Ho et al.)", f"{RAMAN_HANDOFF_DIR}/Untitled-3.1_cnn_myself.ipynb"),
    ("ViT-1D",          f"{RAMAN_HANDOFF_DIR}/Untitled-5.1_transformer_myself.ipynb"),
]

FIGURE4_MODEL_OUTPUT_DIRS: List[Tuple[str, str]] = [
    ("PCA-SVM", f"{RAMAN_HANDOFF_DIR}/_PCA_SVM_v7_target_only_pca3"),
    ("CNN", f"{RAMAN_HANDOFF_DIR}/_CNN_raman_v3_mixture"),
    ("ViT-1D", f"{RAMAN_HANDOFF_DIR}/_ViT1D_raman_v2_mixture"),
]
FIGURE4_OUTPUT_DIR = f"{RAMAN_HANDOFF_DIR}/figure4_outputs"
FIGURE5_OUTPUT_DIR = f"{RAMAN_HANDOFF_DIR}/figure5_outputs"
FIGURE5_MODEL_SUMMARY_CSV = os.path.join(FIGURE5_OUTPUT_DIR, "figure5_model_average_probabilities.csv")
FIGURE5_SVM_TRACE_CSV = os.path.join(FIGURE5_OUTPUT_DIR, "figure5_svm_trace.csv")
REAL_SCAN_TARGET_THRESHOLD = 0.40
REAL_SCAN_WEAK_R6G_THRESHOLD = 0.25
FIGURE5_PARSE_FAILURES = 40

# Print each script/notebook's full output while it runs?
STREAM_LIVE = False

# False: only combine metrics already exported by the three model notebooks.
# True: execute/retrain all three model notebooks before combining results.
RUN_MODEL_NOTEBOOKS = False

# Notebook execution timeout (seconds per cell). Increase if your training
# takes longer than 30 minutes per cell.
NOTEBOOK_CELL_TIMEOUT = 3600   # 1 hour


# =========================================================
# Figure out where to look for the model scripts
# =========================================================
def _resolve_base_dir() -> str:
    try:
        return os.path.dirname(os.path.abspath(__file__))
    except NameError:
        return os.getcwd()


# =========================================================
# Sanity check: peek inside a notebook to verify split ratios
# =========================================================
def _check_notebook_split(nb_path: str, label: str) -> None:
    """Read a .ipynb file and search for TRAIN_RATIO / VAL_RATIO / TEST_RATIO
    in the source cells. Print what we find so the user can verify the notebook
    has the correct split before waiting 10+ minutes for training."""
    if not os.path.exists(nb_path):
        return
    ext = os.path.splitext(nb_path)[1].lower()
    if ext != ".ipynb":
        return  # skip .py files, they print their own split info
    try:
        with open(nb_path, "r", encoding="utf-8") as f:
            nb = json.load(f)
    except Exception:
        return

    # Collect all source code from code cells
    all_source = ""
    for cell in nb.get("cells", []):
        if cell.get("cell_type") == "code":
            src = cell.get("source", "")
            if isinstance(src, list):
                src = "".join(src)
            all_source += src + "\n"

    # Look for ratio assignments
    train_ratio = re.search(r"TRAIN_RATIO\s*=\s*([\d.]+)", all_source)
    val_ratio = re.search(r"VAL_RATIO\s*=\s*([\d.]+)", all_source)
    test_ratio = re.search(r"TEST_RATIO\s*=\s*([\d.]+)", all_source)

    tr = train_ratio.group(1) if train_ratio else "?"
    va = val_ratio.group(1) if val_ratio else "?"
    te = test_ratio.group(1) if test_ratio else "?"

    print(f"  {label:20s}  split={tr}/{va}/{te}  "
          f"last_modified={time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(os.path.getmtime(nb_path)))}")

    # Warn if split doesn't look like 70/10/20
    expected = {"0.70", "0.7"}
    if train_ratio and train_ratio.group(1) not in expected:
        print(f"    *** WARNING: {label} TRAIN_RATIO={tr}, expected 0.70! "
              f"Did you save the notebook after editing? ***")


# =========================================================
# Parsing classification_report output
# =========================================================
_REPORT_HEADER_RE = re.compile(r"\s*precision\s+recall\s+f1-score\s+support", re.IGNORECASE)
_MACRO_AVG_RE = re.compile(
    r"^\s*macro\s+avg\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+(\d+)\s*$",
    re.IGNORECASE | re.MULTILINE,
)
_WEIGHTED_AVG_RE = re.compile(
    r"^\s*weighted\s+avg\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+(\d+)\s*$",
    re.IGNORECASE | re.MULTILINE,
)
_ACCURACY_RE = re.compile(
    r"^\s*accuracy\s+([\d.]+)\s+(\d+)\s*$",
    re.IGNORECASE | re.MULTILINE,
)


def parse_classification_report(stdout_text: str) -> Optional[Dict[str, float]]:
    if not stdout_text:
        return None
    headers = list(_REPORT_HEADER_RE.finditer(stdout_text))
    if not headers:
        return None
    block_text = stdout_text[headers[-1].start():]

    macro_match = _MACRO_AVG_RE.search(block_text)
    weighted_match = _WEIGHTED_AVG_RE.search(block_text)
    accuracy_match = _ACCURACY_RE.search(block_text)

    if macro_match is None and weighted_match is None and accuracy_match is None:
        return None

    out: Dict[str, float] = {}
    if accuracy_match is not None:
        out["accuracy"] = float(accuracy_match.group(1))
        out["support"] = int(accuracy_match.group(2))
    if macro_match is not None:
        out["precision_macro"] = float(macro_match.group(1))
        out["recall_macro"] = float(macro_match.group(2))
        out["f1_macro"] = float(macro_match.group(3))
        if "support" not in out:
            out["support"] = int(macro_match.group(4))
    if weighted_match is not None:
        out["precision_weighted"] = float(weighted_match.group(1))
        out["recall_weighted"] = float(weighted_match.group(2))
        out["f1_weighted"] = float(weighted_match.group(3))

    return out if out else None


# =========================================================
# Runners
# =========================================================
def _run_subprocess_capture(cmd: List[str], cwd: Optional[str], stream_live: bool) -> Tuple[str, int]:
    """Run a subprocess command, stream + capture stdout (with stderr merged)."""
    child_env = os.environ.copy()
    child_env["PYTHONIOENCODING"] = "utf-8"
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        cwd=cwd,
        env=child_env,
    )
    captured: List[str] = []
    assert proc.stdout is not None
    for line in proc.stdout:
        captured.append(line)
        if stream_live:
            sys.stdout.write(line)
            sys.stdout.flush()
    proc.wait()
    return "".join(captured), proc.returncode


def _extract_text_from_executed_notebook(nb_path: str) -> str:
    """Read an executed .ipynb file and concatenate all the printed text outputs
    from each cell, so we can search for classification_report inside it."""
    if not os.path.exists(nb_path):
        return ""
    try:
        with open(nb_path, "r", encoding="utf-8") as f:
            nb = json.load(f)
    except Exception as e:
        print(f"[WARN] Could not parse executed notebook {nb_path}: {e}")
        return ""

    chunks: List[str] = []
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        for out in cell.get("outputs", []):
            otype = out.get("output_type")
            if otype == "stream":
                text = out.get("text", "")
                if isinstance(text, list):
                    text = "".join(text)
                chunks.append(text)
            elif otype in ("execute_result", "display_data"):
                data = out.get("data", {})
                txt = data.get("text/plain", "")
                if isinstance(txt, list):
                    txt = "".join(txt)
                chunks.append(txt)
            elif otype == "error":
                tb = out.get("traceback", [])
                if isinstance(tb, list):
                    chunks.append("\n".join(tb))
    return "\n".join(chunks)


def run_one(script_path: str, label: str, stream_live: bool) -> Tuple[str, float]:
    """Run a .py file or a .ipynb file. Return (captured_text, elapsed_seconds)."""
    if not os.path.exists(script_path):
        raise FileNotFoundError(f"Script not found: {script_path}")

    print("\n" + "#" * 78)
    print(f"# RUNNING: {label}   ({script_path})")
    print("#" * 78, flush=True)

    ext = os.path.splitext(script_path)[1].lower()
    cwd = os.path.dirname(os.path.abspath(script_path)) or None
    t0 = time.time()

    if ext == ".py":
        cmd = [sys.executable, "-u", script_path]
        captured, rc = _run_subprocess_capture(cmd, cwd, stream_live)
        if rc != 0:
            print(f"\n[WARN] {label} exited with code {rc}")
        return captured, time.time() - t0

    if ext == ".ipynb":
        base, _ = os.path.splitext(os.path.basename(script_path))
        executed_name = f"{base}_executed.ipynb"
        executed_path = os.path.join(cwd or ".", executed_name)

        cmd = [
            sys.executable, "-m", "jupyter", "nbconvert",
            "--to", "notebook",
            "--execute",
            "--ExecutePreprocessor.timeout", str(NOTEBOOK_CELL_TIMEOUT),
            "--output", executed_name,
            script_path,
        ]
        nbconvert_log, rc = _run_subprocess_capture(cmd, cwd, stream_live)
        elapsed = time.time() - t0

        if rc != 0:
            print(f"\n[WARN] {label} (notebook) failed to execute.")
            print("----- last nbconvert output -----")
            print(nbconvert_log[-8000:])
            print("----- end nbconvert output -----")

        captured_cell_text = _extract_text_from_executed_notebook(executed_path)

        if stream_live and captured_cell_text:
            print("\n----- Captured cell output from executed notebook -----")
            sys.stdout.write(captured_cell_text)
            sys.stdout.flush()
            print("\n----- end captured cell output -----")

        combined = nbconvert_log + "\n" + captured_cell_text
        return combined, elapsed

    print(f"\n[WARN] {label}: unsupported file extension '{ext}'. "
          f"Expected .py or .ipynb.")
    return "", time.time() - t0


# =========================================================
# Pretty-print the final comparison table
# =========================================================
def print_comparison_table(results: List[Dict]) -> None:
    if not results:
        print("\nNo results parsed.")
        return

    headers = ["Model", "Accuracy", "Prec(macro)", "Rec(macro)", "F1(macro)",
               "F1(weighted)", "Support", "Time(s)"]
    col_widths = [max(len(h), 16) for h in headers]
    col_widths[0] = max(col_widths[0], max(len(r["label"]) for r in results) + 2)

    def fmt_row(cells: List[str]) -> str:
        return "  ".join(c.ljust(w) for c, w in zip(cells, col_widths))

    total_width = sum(col_widths) + 2 * (len(col_widths) - 1)
    print("\n" + "=" * total_width)
    print("FINAL COMPARISON (test-set metrics from each model's classification_report)")
    print("=" * total_width)
    print(fmt_row(headers))
    print("-" * total_width)
    for r in results:
        m = r.get("metrics") or {}
        cells = [
            r["label"],
            f"{m.get('accuracy', float('nan')):.4f}" if "accuracy" in m else "n/a",
            f"{m.get('precision_macro', float('nan')):.4f}" if "precision_macro" in m else "n/a",
            f"{m.get('recall_macro',    float('nan')):.4f}" if "recall_macro"    in m else "n/a",
            f"{m.get('f1_macro',        float('nan')):.4f}" if "f1_macro"        in m else "n/a",
            f"{m.get('f1_weighted',     float('nan')):.4f}" if "f1_weighted"     in m else "n/a",
            f"{m.get('support', 0)}"     if "support" in m else "n/a",
            f"{r['elapsed']:.1f}",
        ]
        print(fmt_row(cells))

    # Verify all models used the same test set size
    supports = [r["metrics"]["support"] for r in results
                if r.get("metrics") and "support" in r["metrics"]]
    if supports and len(set(supports)) > 1:
        print(f"\n*** WARNING: Support values differ across models: {supports}")
        print(f"    This means the test sets are NOT the same size!")
        print(f"    Check that all notebooks use the same split ratios and seed. ***")
    elif supports:
        print(f"\n  All models evaluated on the same {supports[0]} test samples.")


# =========================================================
# Figure 4 standardized aggregation and plotting
# =========================================================
def _read_csv_rows(path: str) -> List[Dict[str, str]]:
    with open(path, "r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def _write_csv_rows(path: str, rows: List[Dict[str, str]]) -> None:
    if not rows:
        raise ValueError(f"No rows to write: {path}")
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def _annotate_heatmap(ax, matrix: np.ndarray, percent: bool = False) -> None:
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = float(matrix[i, j])
            label = f"{value * 100:.1f}%" if percent else f"{value:.3f}"
            color = "white" if value >= 0.82 else "black"
            ax.text(j, i, label, ha="center", va="center", color=color, fontsize=10)


def build_figure4_outputs() -> None:
    os.makedirs(FIGURE4_OUTPUT_DIR, exist_ok=True)
    overall_rows: List[Dict[str, str]] = []
    per_target_rows: List[Dict[str, str]] = []

    for expected_model, model_dir in FIGURE4_MODEL_OUTPUT_DIRS:
        overall_path = os.path.join(model_dir, "figure4_overall_metrics.csv")
        per_target_path = os.path.join(model_dir, "figure4_per_target_metrics.csv")
        missing = [p for p in (overall_path, per_target_path) if not os.path.exists(p)]
        if missing:
            raise FileNotFoundError(
                f"Missing Figure 4 exports for {expected_model}: {missing}. "
                "Run the three modified model notebooks first."
            )
        model_overall = _read_csv_rows(overall_path)
        model_targets = _read_csv_rows(per_target_path)
        if len(model_overall) != 1:
            raise ValueError(f"Expected one overall row for {expected_model}, got {len(model_overall)}")
        if model_overall[0]["model"] != expected_model:
            raise ValueError(f"Model name mismatch in {overall_path}")
        overall_rows.extend(model_overall)
        per_target_rows.extend(model_targets)

    n_tests = {int(float(row["n_test"])) for row in overall_rows}
    n_mixtures = {int(float(row["n_mixture"])) for row in overall_rows}
    thresholds = {float(row["benchmark_threshold"]) for row in overall_rows}
    if len(n_tests) != 1:
        raise ValueError(f"The models do not have the same test-set size: {sorted(n_tests)}")
    if len(n_mixtures) != 1:
        raise ValueError(f"The models do not have the same multi-target subset size: {sorted(n_mixtures)}")
    if thresholds != {0.5}:
        raise ValueError(f"Figure 4 benchmark threshold must be 0.50, got {sorted(thresholds)}")

    overall_csv = os.path.join(FIGURE4_OUTPUT_DIR, "overall_metrics.csv")
    per_target_csv = os.path.join(FIGURE4_OUTPUT_DIR, "per_target_metrics.csv")
    _write_csv_rows(overall_csv, overall_rows)
    _write_csv_rows(per_target_csv, per_target_rows)

    model_order = [name for name, _ in FIGURE4_MODEL_OUTPUT_DIRS]
    overall_by_model = {row["model"]: row for row in overall_rows}
    overall_metrics = [
        ("target_macro_f1", "Target macro-F1"),
        ("target_macro_roc_auc", "Target macro ROC-AUC"),
        ("r6g_recall", "R6G recall"),
        ("mixture_exact_match", "Mixture exact-match"),
    ]
    overall_matrix = np.array([
        [float(overall_by_model[m][key]) for key, _ in overall_metrics]
        for m in model_order
    ], dtype=float)

    target_order = [("r6g", "R6G"), ("methylene_blue", "Methylene blue"), ("adenine", "Adenine")]
    target_lookup = {(row["model"], row["target"]): row for row in per_target_rows}
    f1_matrix = np.array([
        [float(target_lookup[(model, target_key)]["f1"]) for target_key, _ in target_order]
        for model in model_order
    ], dtype=float)

    # Representative portable-scan summary used only in panel D.  These
    # values are sample-level averages from the same MB + R6G real scan,
    # not additional held-out benchmark samples.
    if not os.path.exists(FIGURE5_MODEL_SUMMARY_CSV):
        raise FileNotFoundError(f"Missing real-scan summary: {FIGURE5_MODEL_SUMMARY_CSV}")
    real_scan_rows = _read_csv_rows(FIGURE5_MODEL_SUMMARY_CSV)
    real_scan_by_model = {row["model"]: row for row in real_scan_rows}
    missing_real_models = [m for m in model_order if m not in real_scan_by_model]
    if missing_real_models:
        raise ValueError(f"Real-scan summary is missing models: {missing_real_models}")
    real_scan_matrix = np.array([
        [float(real_scan_by_model[model][target_key]) for target_key, _ in target_order]
        for model in model_order
    ], dtype=float)
    real_scan_csv = os.path.join(FIGURE4_OUTPUT_DIR, "real_scan_model_summary.csv")
    _write_csv_rows(real_scan_csv, [real_scan_by_model[m] for m in model_order])
    n_test = next(iter(n_tests)); n_mixture = next(iter(n_mixtures))
    benchmark_note = f"Benchmark threshold = 0.50; test n={n_test}; multi-target subset n={n_mixture}."
    model_colors = ["#356B8C", "#35B779", "#F2C14E"]

    def _draw_workflow(ax):
        from matplotlib.patches import FancyBboxPatch
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
        boxes = [
            (0.03, 0.35, 0.18, 0.30, "Raman spectra\npure + synthetic\nmixtures", "#EAF1F6"),
            (0.29, 0.35, 0.20, 0.30, "Preprocessing\ndark subtraction\nbaseline + scaling", "#E8F4EC"),
            (0.57, 0.69, 0.18, 0.20, "PCA-SVM\nPCA + peak features", "#FFF2CF"),
            (0.57, 0.40, 0.18, 0.20, "1D-CNN\nSE-ResNet\nsigmoid labels", "#EEE9F6"),
            (0.57, 0.11, 0.18, 0.20, "ViT-1D\npatch tokens\nsigmoid labels", "#E9F0F8"),
            (0.83, 0.31, 0.14, 0.38, "Target analytes\nR6G\nMB\nadenine", "#F8E9E8"),
        ]
        for x, y, w, h, label, face in boxes:
            patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.008", facecolor=face, edgecolor="#333333", linewidth=1.0)
            ax.add_patch(patch); ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=9)
        arrow = dict(arrowstyle="-|>", color="#333333", lw=1.1, mutation_scale=11)
        ax.annotate("", xy=(0.29, 0.50), xytext=(0.21, 0.50), arrowprops=arrow)
        for y in (0.79, 0.50, 0.21):
            ax.annotate("", xy=(0.57, y), xytext=(0.49, 0.50), arrowprops=arrow)
            ax.annotate("", xy=(0.83, 0.50), xytext=(0.75, y), arrowprops=arrow)
        ax.text(0.50, 0.015, "DI water is treated as solvent/background for final target reporting.", ha="center", va="bottom", fontsize=8, color="#555555")

    def _draw_overall(ax, cax=None):
        im = ax.imshow(overall_matrix, cmap="YlGnBu", vmin=0.55, vmax=1.00, aspect="auto")
        ax.set_xticks(np.arange(len(overall_metrics)))
        ax.set_xticklabels([label for _, label in overall_metrics], rotation=22, ha="right")
        ax.set_yticks(np.arange(len(model_order))); ax.set_yticklabels(model_order)
        ax.set_title("Overall target-analyte performance")
        _annotate_heatmap(ax, overall_matrix, percent=True)
        if cax is None:
            cbar = ax.figure.colorbar(im, ax=ax, fraction=0.045, pad=0.035)
        else:
            cbar = ax.figure.colorbar(im, cax=cax)
        cbar.set_label("Score")
        return im

    def _draw_f1(ax):
        x = np.arange(len(target_order)); width = 0.24
        for i, model in enumerate(model_order):
            bars = ax.bar(x + (i - 1) * width, f1_matrix[i], width, label=model, color=model_colors[i], edgecolor="#333333", linewidth=0.6)
            ax.bar_label(bars, labels=[f"{v:.2f}" for v in f1_matrix[i]], padding=3, fontsize=8)
        ax.set_xticks(x); ax.set_xticklabels([label for _, label in target_order])
        ax.set_ylabel("F1 score"); ax.set_ylim(0.65, 1.025); ax.set_yticks(np.arange(0.7, 1.01, 0.1))
        ax.set_title("Per-target F1 score")
        ax.grid(axis="y", alpha=0.25, linewidth=0.7); ax.set_axisbelow(True)
        ax.legend(loc="lower right", frameon=False, fontsize=9)

    def _draw_real_scan(ax):
        x = np.arange(len(target_order)); width = 0.24
        for i, model in enumerate(model_order):
            bars = ax.bar(
                x + (i - 1) * width, real_scan_matrix[i], width,
                label=model, color=model_colors[i], edgecolor="#333333", linewidth=0.6,
            )
            ax.bar_label(bars, labels=[f"{v:.2f}" for v in real_scan_matrix[i]], padding=3, fontsize=8)
        ax.axhline(REAL_SCAN_TARGET_THRESHOLD, color="#555555", linestyle="--", linewidth=1.0, label="target threshold")
        ax.hlines(REAL_SCAN_WEAK_R6G_THRESHOLD, -0.45, 0.45, color="#888888", linestyle=":", linewidth=1.0, label="weak-R6G threshold")
        ax.set_xticks(x); ax.set_xticklabels([label for _, label in target_order])
        ax.set_ylabel("Average OvR probability"); ax.set_ylim(0, 1.08)
        ax.set_title("Representative real-scan sample-level probabilities")
        ax.grid(axis="y", alpha=0.25, linewidth=0.7); ax.set_axisbelow(True)
        ax.legend(loc="center right", frameon=False, fontsize=8)
        ax.text(0.0, -0.22, "Same MB + R6G scan for all models; one representative scan.", transform=ax.transAxes, fontsize=8, ha="left", va="top")

    # Panel A: workflow.
    fig, ax = plt.subplots(figsize=(11.0, 3.6))
    _draw_workflow(ax); fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(os.path.join(FIGURE4_OUTPUT_DIR, f"figure4a_workflow.{ext}"), dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

    # Panel B: overall comparison, including macro ROC-AUC.
    fig = plt.figure(figsize=(10.4, 5.2), constrained_layout=True)
    grid = fig.add_gridspec(1, 2, width_ratios=[1, 0.035], wspace=0.05)
    ax = fig.add_subplot(grid[0, 0]); cax = fig.add_subplot(grid[0, 1])
    _draw_overall(ax, cax)
    ax.text(0.0, -0.35, benchmark_note, transform=ax.transAxes, fontsize=9, ha="left", va="top")
    for ext in ("png", "pdf", "svg"):
        fig.savefig(os.path.join(FIGURE4_OUTPUT_DIR, f"figure4b_model_comparison.{ext}"), dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

    # Panel C: grouped per-target F1 bars, matching the manuscript layout.
    fig, ax = plt.subplots(figsize=(7.4, 5.0))
    _draw_f1(ax); fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(os.path.join(FIGURE4_OUTPUT_DIR, f"figure4c_per_target_f1.{ext}"), dpi=300, bbox_inches="tight")
        fig.savefig(os.path.join(FIGURE4_OUTPUT_DIR, f"figure4c_per_target_metrics.{ext}"), dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

    # Panel D: sample-level comparison on one representative portable scan.
    fig, ax = plt.subplots(figsize=(8.6, 5.2))
    _draw_real_scan(ax); fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(os.path.join(FIGURE4_OUTPUT_DIR, f"figure4d_real_scan_comparison.{ext}"), dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

    # Complete A/B/C/D manuscript figure.
    fig = plt.figure(figsize=(17.0, 9.2), constrained_layout=True)
    outer = fig.add_gridspec(2, 1, height_ratios=[0.86, 1.30])
    ax_a = fig.add_subplot(outer[0, 0]); _draw_workflow(ax_a)
    lower = outer[1, 0].subgridspec(1, 3, width_ratios=[1.25, 1.00, 1.15], wspace=0.18)
    ax_b = fig.add_subplot(lower[0, 0]); _draw_overall(ax_b)
    ax_c = fig.add_subplot(lower[0, 1]); _draw_f1(ax_c)
    ax_d = fig.add_subplot(lower[0, 2]); _draw_real_scan(ax_d)
    ax_a.text(-0.015, 1.02, "A", transform=ax_a.transAxes, fontsize=15, fontweight="bold", va="bottom")
    ax_b.text(-0.22, 1.06, "B", transform=ax_b.transAxes, fontsize=15, fontweight="bold", va="bottom")
    ax_c.text(-0.14, 1.06, "C", transform=ax_c.transAxes, fontsize=15, fontweight="bold", va="bottom")
    ax_d.text(-0.14, 1.06, "D", transform=ax_d.transAxes, fontsize=15, fontweight="bold", va="bottom")
    ax_b.text(-0.02, -0.34, benchmark_note, transform=ax_b.transAxes, fontsize=8.2, ha="left", va="top")
    for stem in ("figure4_ABCD", "figure4_ABCD_manuscript", "figure4_manuscript"):
        for ext in ("png", "pdf", "svg"):
            fig.savefig(os.path.join(FIGURE4_OUTPUT_DIR, f"{stem}.{ext}"), dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

    print("\nFigure 4 exports complete:")
    print(f"  {overall_csv}")
    print(f"  {per_target_csv}")
    print(f"  {os.path.join(FIGURE4_OUTPUT_DIR, 'figure4a_workflow.png')}")
    print(f"  {os.path.join(FIGURE4_OUTPUT_DIR, 'figure4b_model_comparison.png')}")
    print(f"  {os.path.join(FIGURE4_OUTPUT_DIR, 'figure4c_per_target_f1.png')}")
    print(f"  {os.path.join(FIGURE4_OUTPUT_DIR, 'figure4d_real_scan_comparison.png')}")
    print(f"  {os.path.join(FIGURE4_OUTPUT_DIR, 'figure4_ABCD_manuscript.png')}")


# =========================================================
# Figure 5: time-resolved real-scan reporting
# =========================================================
def build_figure5_outputs() -> None:
    os.makedirs(FIGURE5_OUTPUT_DIR, exist_ok=True)
    for required_path in (FIGURE5_SVM_TRACE_CSV, FIGURE5_MODEL_SUMMARY_CSV):
        if not os.path.exists(required_path):
            raise FileNotFoundError(f"Missing Figure 5 input: {required_path}")

    trace_rows = _read_csv_rows(FIGURE5_SVM_TRACE_CSV)
    if not trace_rows:
        raise ValueError(f"No rows in {FIGURE5_SVM_TRACE_CSV}")
    scan_target_order = [("r6g", "R6G"), ("methylene_blue", "Methylene blue"), ("adenine", "Adenine")]
    sequence = np.arange(1, len(trace_rows) + 1, dtype=int)
    spectrum_probs = np.column_stack([
        np.array([float(row[key]) for row in trace_rows], dtype=float)
        for key, _ in scan_target_order
    ])
    cumulative_probs = np.cumsum(spectrum_probs, axis=0) / sequence[:, None]

    sample_trace_rows = []
    for i, n_seen in enumerate(sequence):
        sample_trace_rows.append({
            "accepted_spectrum_count": int(n_seen),
            "r6g": float(cumulative_probs[i, 0]),
            "methylene_blue": float(cumulative_probs[i, 1]),
            "adenine": float(cumulative_probs[i, 2]),
        })
    sample_trace_csv = os.path.join(FIGURE5_OUTPUT_DIR, "figure5_sample_level_trace.csv")
    _write_csv_rows(sample_trace_csv, sample_trace_rows)

    target_colors = ["#356B8C", "#35B779", "#F2C14E"]

    def _draw_streaming_workflow(ax):
        from matplotlib.patches import FancyBboxPatch
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
        boxes = [
            (0.02, 0.35, 0.15, 0.30, "Portable Raman\nscan", "#EAF1F6"),
            (0.22, 0.35, 0.16, 0.30, "Parse + dark\nsubtraction", "#E8F4EC"),
            (0.43, 0.31, 0.16, 0.38, f"Quality filter\n{len(trace_rows)} accepted\n{FIGURE5_PARSE_FAILURES} parse failures", "#FFF2CF"),
            (0.64, 0.31, 0.17, 0.38, "Update cumulative\nsample-level\nprobabilities", "#EEE9F6"),
            (0.86, 0.31, 0.12, 0.38, "Final decision\nMB + R6G", "#F8E9E8"),
        ]
        for x, y, w, h, label, face in boxes:
            patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.008", facecolor=face, edgecolor="#333333", linewidth=1.0)
            ax.add_patch(patch); ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=9)
        arrow = dict(arrowstyle="-|>", color="#333333", lw=1.1, mutation_scale=11)
        for x0, x1 in ((0.17, 0.22), (0.38, 0.43), (0.59, 0.64), (0.81, 0.86)):
            ax.annotate("", xy=(x1, 0.50), xytext=(x0, 0.50), arrowprops=arrow)
        ax.text(0.50, 0.06, "Probabilities are updated after each accepted spectrum.", ha="center", va="center", fontsize=8, color="#555555")

    def _draw_spectrum_trace(ax):
        for j, (_, label) in enumerate(scan_target_order):
            ax.plot(sequence, spectrum_probs[:, j], marker="o", markersize=2.1, linewidth=1.0, color=target_colors[j], label=label)
        ax.axhline(REAL_SCAN_TARGET_THRESHOLD, color="#555555", linestyle="--", linewidth=1.0, label="target threshold")
        ax.axhline(REAL_SCAN_WEAK_R6G_THRESHOLD, color="#888888", linestyle=":", linewidth=1.0, label="weak-R6G threshold")
        ax.set_xlabel("Accepted spectrum sequence")
        ax.set_ylabel("OvR probability"); ax.set_ylim(0, 1.05); ax.set_xlim(1, len(sequence))
        ax.set_title("PCA-SVM spectrum-level probabilities")
        ax.grid(alpha=0.22, linewidth=0.7); ax.set_axisbelow(True)
        ax.legend(loc="lower right", frameon=False, fontsize=8, ncol=2)

    def _draw_sample_trace(ax):
        for j, (_, label) in enumerate(scan_target_order):
            ax.plot(sequence, cumulative_probs[:, j], linewidth=2.0, color=target_colors[j], label=label)
        ax.axhline(REAL_SCAN_TARGET_THRESHOLD, color="#555555", linestyle="--", linewidth=1.0, label="target threshold")
        ax.axhline(REAL_SCAN_WEAK_R6G_THRESHOLD, color="#888888", linestyle=":", linewidth=1.0, label="weak-R6G threshold")
        ax.set_xlabel("Accepted spectra accumulated")
        ax.set_ylabel("Cumulative sample-level probability")
        ax.set_ylim(0, 1.05); ax.set_xlim(1, len(sequence) * 1.13)
        ax.set_title("Sample-level probabilities during acquisition")
        ax.grid(alpha=0.22, linewidth=0.7); ax.set_axisbelow(True)
        for j, (_, label) in enumerate(scan_target_order):
            final_value = cumulative_probs[-1, j]
            ax.annotate(f"{label} {final_value:.2f}", xy=(sequence[-1], final_value), xytext=(5, 0), textcoords="offset points", va="center", fontsize=8, color=target_colors[j])
        ax.legend(loc="center right", frameon=False, fontsize=8)

    fig, ax = plt.subplots(figsize=(11.5, 3.5))
    _draw_streaming_workflow(ax); fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(os.path.join(FIGURE5_OUTPUT_DIR, f"figure5a_streaming_workflow.{ext}"), dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    _draw_spectrum_trace(ax); fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(os.path.join(FIGURE5_OUTPUT_DIR, f"figure5b_spectrum_level_trace.{ext}"), dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    _draw_sample_trace(ax); fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(os.path.join(FIGURE5_OUTPUT_DIR, f"figure5c_sample_level_trace.{ext}"), dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

    fig = plt.figure(figsize=(13.5, 8.0), constrained_layout=True)
    outer = fig.add_gridspec(2, 2, height_ratios=[0.72, 1.28])
    ax_a = fig.add_subplot(outer[0, :]); _draw_streaming_workflow(ax_a)
    ax_b = fig.add_subplot(outer[1, 0]); _draw_spectrum_trace(ax_b)
    ax_c = fig.add_subplot(outer[1, 1]); _draw_sample_trace(ax_c)
    ax_a.text(-0.015, 1.02, "A", transform=ax_a.transAxes, fontsize=15, fontweight="bold", va="bottom")
    ax_b.text(-0.10, 1.02, "B", transform=ax_b.transAxes, fontsize=15, fontweight="bold", va="bottom")
    ax_c.text(-0.10, 1.02, "C", transform=ax_c.transAxes, fontsize=15, fontweight="bold", va="bottom")
    for stem in ("figure5_ABC_time_resolved", "figure5_ABC_time_resolved_manuscript", "figure5_manuscript"):
        for ext in ("png", "pdf", "svg"):
            fig.savefig(os.path.join(FIGURE5_OUTPUT_DIR, f"{stem}.{ext}"), dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

    print("\nFigure 5 exports complete:")
    print(f"  {sample_trace_csv}")
    print(f"  {os.path.join(FIGURE5_OUTPUT_DIR, 'figure5a_streaming_workflow.png')}")
    print(f"  {os.path.join(FIGURE5_OUTPUT_DIR, 'figure5b_spectrum_level_trace.png')}")
    print(f"  {os.path.join(FIGURE5_OUTPUT_DIR, 'figure5c_sample_level_trace.png')}")
    print(f"  {os.path.join(FIGURE5_OUTPUT_DIR, 'figure5_ABC_time_resolved_manuscript.png')}")


# =========================================================
# Main
# =========================================================
def main() -> None:
    base_dir = _resolve_base_dir()
    print(f"[run_all] Base directory for relative paths: {base_dir}")

    resolved: List[Tuple[str, str]] = []
    for label, path in MODEL_SCRIPTS:
        full = path if os.path.isabs(path) else os.path.join(base_dir, path)
        resolved.append((label, full))

    missing = [(label, p) for label, p in resolved if not os.path.exists(p)]
    if missing:
        print("\n[WARN] These scripts/notebooks were not found and will be skipped:")
        for label, p in missing:
            print(f"       - {label}: {p}")

    # ---- Pre-flight sanity check: peek inside each notebook ----
    print("\n" + "=" * 78)
    print("PRE-FLIGHT CHECK: verifying notebook split ratios")
    print("=" * 78)
    any_warning = False
    for label, path in resolved:
        if os.path.exists(path):
            _check_notebook_split(path, label)

    print("")

    if RUN_MODEL_NOTEBOOKS:
        for label, path in resolved:
            if not os.path.exists(path):
                continue
            _, elapsed = run_one(path, label, STREAM_LIVE)
            print(f"\n  Completed {label} in {elapsed:.1f} s")
    else:
        print("\nRUN_MODEL_NOTEBOOKS=False: skipping model execution and aggregating existing exports.")

    build_figure4_outputs()
    build_figure5_outputs()


if __name__ == "__main__":
    main()

[run_all] Base directory for relative paths: c:\Users\Jeremy666\OneDrive\Documents\laser_test\build\Desktop_Qt_6_6_3_MSVC2019_64bit-Debug

PRE-FLIGHT CHECK: verifying notebook split ratios & classification_report
  PCA-SVM               split=0.70/0.10/0.20  classification_report=YES  last_modified=2026-04-30 01:30:27
  CNN (Ho et al.)       split=0.70/0.10/0.20  classification_report=YES  last_modified=2026-04-30 12:10:27
  ViT-1D                split=0.70/0.10/0.20  classification_report=YES  last_modified=2026-04-30 12:06:13


##############################################################################
# RUNNING: PCA-SVM   (C:\Users\Jeremy666\OneDrive\Documents\laser_test\build\Desktop_Qt_6_6_3_MSVC2019_64bit-Debug\Untitled-4_svm.ipynb)
##############################################################################
[NbConvertApp] Converting notebook C:\Users\Jeremy666\OneDrive\Documents\laser_test\build\Desktop_Qt_6_6_3_MSVC2019_64bit-Debug\Untitled-4_svm.ipynb to notebook
C:\Users